[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C34_Agent_Orchestration_Course/05_deploy/05_deploy.ipynb)

# 05 · 部署 Agent（Deployment）

目标：用**纯标准库**从零搭一个 agent 部署骨架——**任务队列 → worker + 状态机 → 重试退避 → 状态持久化 → 幂等/健康检查/优雅停机**，全程 `assert` 验证，**无需 API key**。

路线：FIFO+优先级队列 → 状态机(合法转移) → worker 主循环 → 重试退避(区分可重试) → 持久化(崩溃恢复 running) → 幂等键 → 健康检查 → 把全栈(01-05)拧成一个 agent 服务 → ✏️ 练习 → 📖 答案 → 🧪 真实部署语义胶囊。

> 心智模型：**部署 = 给 agent 装上能 7×24 自动运转的运行时**。任务进队列、worker 不停取、状态机管一生、退避重试、状态落盘防丢、健康探针报死活。假设最坏情况一定发生，为每一种坏情况预先准备好应对。

## 1 · 任务队列：FIFO + 优先级（解耦接收与执行）

队列把『接收任务(producer)』与『执行任务(worker)』解耦：削峰、扩容、可重试。
实现一个最小队列：FIFO `enqueue/dequeue` + 可选优先级（高优先先出）。

In [ ]:
import json, time, heapq
from collections import deque

class TaskQueue:
    '''内存任务队列(真实里换 Redis/RabbitMQ/SQS，接口语义一致)。
       priority 越小越先出；同优先级按入队顺序(FIFO)。'''
    def __init__(self):
        self._heap = []           # (priority, seq, task)
        self._seq = 0
    def enqueue(self, task, priority=5):
        heapq.heappush(self._heap, (priority, self._seq, task))
        self._seq += 1
    def dequeue(self):
        if not self._heap:
            return None
        return heapq.heappop(self._heap)[2]
    def __len__(self):
        return len(self._heap)

q = TaskQueue()
q.enqueue({'id': 't1', 'job': '普通任务A'})            # 默认优先级 5
q.enqueue({'id': 't2', 'job': '普通任务B'})
q.enqueue({'id': 't3', 'job': '紧急任务'}, priority=1)  # 高优先(数字小)
print('队列深度:', len(q))
order = [q.dequeue()['id'] for _ in range(3)]
print('出队顺序:', order)
assert order == ['t3', 't1', 't2']        # 紧急先出，其余 FIFO
assert q.dequeue() is None and len(q) == 0  # 空队列返回 None
print('✅ 任务队列：高优先先出、同优先 FIFO、空队返回 None')

## 2 · 状态机：合法转移 + 拒绝非法

把任务一生建成『有限状态 + 合法转移』。状态机能**拒绝非法转移**(succeeded 不能再变 running)。
用一张转移表守住，比散落的布尔标志可靠得多。

In [ ]:
# 合法转移表: 每个状态 -> 它能去的状态集合
TRANSITIONS = {
    'queued':    {'running'},
    'running':   {'succeeded', 'failed'},
    'failed':    {'queued', 'dead'},    # 失败可重试(回 queued)或判死
    'succeeded': set(),                  # 终态
    'dead':      set(),                  # 终态
}

class Task:
    def __init__(self, tid, job):
        self.id, self.job = tid, job
        self.state = 'queued'
        self.attempts = 0
        self.history = ['queued']
    def transition(self, to_state):
        if to_state not in TRANSITIONS[self.state]:
            raise ValueError(f'非法转移: {self.state} -> {to_state}')
        self.state = to_state
        self.history.append(to_state)    # 记状态变迁史(可观测, 接模块 04)

t = Task('t1', '研究任务')
t.transition('running')
t.transition('succeeded')
print('状态变迁史:', t.history)
assert t.state == 'succeeded' and t.history == ['queued', 'running', 'succeeded']
# 非法转移被拒: succeeded 不能回到 running
try:
    t.transition('running'); raised = False
except ValueError:
    raised = True
assert raised
# 失败->重试(回 queued)是合法的
t2 = Task('t2', 'x'); t2.transition('running'); t2.transition('failed')
t2.transition('queued')        # 合法: 重试
assert t2.state == 'queued'
print('✅ 状态机：合法转移放行、非法转移(succeeded->running)被拒绝')

## 3 · worker 主循环：取任务 → 跑 → 标记状态

worker 不停从队列取任务、驱动状态机：`dequeue → running → 执行 → succeeded/failed`。
把任务表(所有任务+状态)集中管理，便于持久化与健康检查。

In [ ]:
class AgentService:
    '''最小 agent 部署服务: 队列 + 任务表 + worker 主循环。'''
    def __init__(self):
        self.queue = TaskQueue()
        self.tasks = {}            # tid -> Task(集中管理, 供持久化/健康检查)
    def submit(self, tid, job, priority=5):
        task = Task(tid, job)
        self.tasks[tid] = task
        self.queue.enqueue(tid, priority)   # 队列里只放 id
        return tid
    def run_one(self, handler):
        '''worker 处理一个任务。handler(job)->result, 抛异常即失败。'''
        tid = self.queue.dequeue()
        if tid is None:
            return None
        task = self.tasks[tid]
        task.transition('running')
        task.attempts += 1
        try:
            task.result = handler(task.job)
            task.transition('succeeded')
        except Exception as e:
            task.error = str(e)
            task.transition('failed')
        return task
    def run_until_empty(self, handler, max_iter=100):
        done = 0
        while len(self.queue) and done < max_iter:
            self.run_one(handler); done += 1
        return done

svc = AgentService()
svc.submit('t1', '研究A'); svc.submit('t2', '研究B', priority=1); svc.submit('t3', 'BOOM')

def agent_handler(job):
    if job == 'BOOM': raise RuntimeError('agent 跑挂了')
    return f'{job}-完成'

svc.run_until_empty(agent_handler)
for tid, t in svc.tasks.items():
    print(f'{tid}: {t.state:10s} attempts={t.attempts}')
assert svc.tasks['t1'].state == 'succeeded' and svc.tasks['t2'].state == 'succeeded'
assert svc.tasks['t3'].state == 'failed'          # BOOM 失败但被隔离, 不影响其它
assert svc.tasks['t1'].result == '研究A-完成'
print('✅ worker 主循环：取任务->running->执行->succeeded/失败隔离；任务表集中管理')

## 4 · 重试退避：区分可重试 + 指数退避 + 上限

失败先**区分**：瞬时(超时/5xx)可重试，确定性(参数非法)立即判死。
可重试的用**指数退避**`base*2**k`重新入队；超**上限**转 `dead`(死信)。

In [ ]:
class TransientError(Exception):
    '''瞬时错误: 可重试。'''
class PermanentError(Exception):
    '''确定性错误: 不可重试。'''

def backoff_wait(attempt, base=0.0):
    '''第 attempt 次重试前的等待 = base * 2**attempt(演示用 base=0 不真等)。'''
    return base * (2 ** attempt)

def run_with_retry(task, handler, max_retries=3, base=0.0):
    '''带重试退避的执行(顺序模拟). 区分可重试/不可重试; 超上限 -> dead。
       返回 (最终状态, 实际尝试次数)。'''
    for attempt in range(max_retries + 1):
        task.transition('running')
        task.attempts += 1
        try:
            task.result = handler(task.job)
            task.transition('succeeded')
            return 'succeeded', task.attempts
        except PermanentError:
            task.transition('failed'); task.transition('dead')   # 不可重试: 直接判死
            return 'dead', task.attempts
        except TransientError:
            task.transition('failed')
            if attempt == max_retries:
                task.transition('dead')        # 重试用尽 -> 死信
                return 'dead', task.attempts
            time.sleep(backoff_wait(attempt, base))
            task.transition('queued')          # 退避后重新入队

# 情形A: 前 2 次瞬时失败、第 3 次成功
state_box = {'n': 0}
def flaky(job):
    state_box['n'] += 1
    if state_box['n'] <= 2: raise TransientError('暂不可用(5xx)')
    return job + '-ok'
ta = Task('a', '任务')
st, tries = run_with_retry(ta, flaky, max_retries=3)
print(f'瞬时错误: {st}, 尝试 {tries} 次')
assert st == 'succeeded' and tries == 3
# 情形B: 确定性错误立即判死、不重试
tb = Task('b', 'x')
st2, tries2 = run_with_retry(tb, lambda j: (_ for _ in ()).throw(PermanentError('参数非法')))
assert st2 == 'dead' and tries2 == 1          # 只试 1 次就判死
# 情形C: 一直瞬时失败 -> 重试用尽进死信
tc = Task('c', 'x')
st3, tries3 = run_with_retry(tc, lambda j: (_ for _ in ()).throw(TransientError('一直挂')), max_retries=3)
assert st3 == 'dead' and tries3 == 4          # 1 + 3 次重试
print(f'确定性错误: {st2}({tries2}次) | 一直瞬时: {st3}({tries3}次进死信)')
print('✅ 重试退避：瞬时退避重试到成功、确定性立即判死、用尽进死信')

## 5 · 状态持久化：崩溃重启恢复 running 任务

服务一定会重启。把任务表 dump 成 JSON 存盘；重启时 load 回来。
**关键**：崩溃时卡在 `running` 的任务(它的 worker 没了)要**重新入队**(回 queued)等幂等重跑。

In [ ]:
def snapshot(tasks):
    '''把任务表序列化(真实里写 DB/Redis；这里返回可 JSON 化的 dict)。'''
    return {tid: {'id': t.id, 'job': t.job, 'state': t.state,
                  'attempts': t.attempts} for tid, t in tasks.items()}

def restore(snap):
    '''从快照恢复任务表。崩溃时卡在 running 的视为中断 -> 重新入队。'''
    tasks = {}
    requeued = []
    for tid, d in snap.items():
        t = Task(tid, d['job'])
        t.attempts = d['attempts']
        state = d['state']
        if state == 'running':
            state = 'queued'                 # ← 孤儿任务: 重新排队
            requeued.append(tid)
        t.state = state
        t.history = [state]
        tasks[tid] = t
    return tasks, requeued

# 模拟崩溃: 一个 running、一个 succeeded、一个 queued
crashed = {
    'r1': {'id': 'r1', 'job': '研究X', 'state': 'running', 'attempts': 1},   # 崩溃时正在跑
    'r2': {'id': 'r2', 'job': '研究Y', 'state': 'succeeded', 'attempts': 1},
    'r3': {'id': 'r3', 'job': '研究Z', 'state': 'queued', 'attempts': 0},
}
# 验证 JSON 可往返(真正写盘的形状)
assert json.loads(json.dumps(crashed)) == crashed
tasks, requeued = restore(crashed)
print('恢复后状态:', {tid: t.state for tid, t in tasks.items()})
print('被重新入队的孤儿任务:', requeued)
assert tasks['r1'].state == 'queued' and requeued == ['r1']   # running -> 重新排队
assert tasks['r2'].state == 'succeeded'                        # 已完成的保持
assert tasks['r3'].state == 'queued'                           # 排队的保持
print('✅ 持久化恢复：JSON 可往返；崩溃时 running 的孤儿任务被重新入队等重跑')

## 6 · 幂等键 + 健康检查

『至少一次』投递 + 崩溃重跑 -> 任务可能跑多次。**幂等键**保证重复执行不重复副作用。
**健康检查**报告队列深度/各状态数/成功率，让外部探测服务死活。

In [ ]:
class IdempotentRunner:
    '''幂等执行: 同一 key 处理过就返回上次结果, 不重复执行副作用。'''
    def __init__(self):
        self.done = {}            # idempotency_key -> result
        self.side_effects = 0     # 统计真正执行的副作用次数
    def run(self, key, handler, job):
        if key in self.done:
            return self.done[key]   # 已处理 -> 直接返回, 不再执行
        result = handler(job)       # 真正执行(副作用在这)
        self.side_effects += 1
        self.done[key] = result
        return result

runner = IdempotentRunner()
charge = lambda job: f'已扣款 {job}元'
# 同一个任务(同 key)执行 3 次, 副作用只发生 1 次
r1 = runner.run('order-42', charge, 100)
r2 = runner.run('order-42', charge, 100)   # 重复! 不再扣款
r3 = runner.run('order-42', charge, 100)
assert r1 == r2 == r3 == '已扣款 100元'
assert runner.side_effects == 1            # 关键: 只扣了一次!
# 不同 key 才真正执行
runner.run('order-43', charge, 50)
assert runner.side_effects == 2
print('同 key 跑 3 次, 实际扣款次数:', runner.side_effects, '(幂等防重复扣款)')
print('✅ 幂等键：重复投递/重跑下, 副作用只发生一次')

In [ ]:
def health_check(tasks, queue, max_queue=1000):
    '''健康检查: liveness(进程活着) + readiness(能接流量) + 关键指标。'''
    from collections import Counter
    states = Counter(t.state for t in tasks.values())
    n_done = states['succeeded'] + states['dead']
    n_failed_terminal = states['dead']
    error_rate = (n_failed_terminal / n_done) if n_done else 0.0
    ready = len(queue) < max_queue and error_rate < 0.5   # 队列没爆 + 错误率不高
    return {'live': True, 'ready': ready, 'queue_depth': len(queue),
            'states': dict(states), 'error_rate': round(error_rate, 3)}

# 健康场景
svc2 = AgentService()
for i in range(5): svc2.submit(f't{i}', 'job')
svc2.run_until_empty(lambda j: 'ok')
h = health_check(svc2.tasks, svc2.queue)
print('健康检查:', h)
assert h['live'] is True and h['ready'] is True
assert h['queue_depth'] == 0 and h['states']['succeeded'] == 5
assert h['error_rate'] == 0.0
print('✅ 健康检查：报告 live/ready/队列深度/各状态数/错误率 —— 自动运维的眼睛')

---
## ✏️ 练习 1：是否该重试（区分错误类型）

重试的第一道闸是**区分可重试性**——确定性错误重试无用。

实现 `should_retry(error_type, status_code, attempt, max_retries)`：返回是否应重试。规则：① `attempt >= max_retries` → False(用尽)；② `error_type=='timeout'` → True；③ `status_code` 在 500..599 → True；④ `status_code` 在 400..499 → False(客户端错误不可重试)；⑤ 其它 → False。

In [ ]:
def should_retry(error_type, status_code, attempt, max_retries):
    # TODO: 按题面规则返回 bool(注意先判用尽)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert should_retry('timeout', None, attempt=0, max_retries=3) is True    # 超时可重试
assert should_retry(None, 503, attempt=1, max_retries=3) is True          # 5xx 可重试
assert should_retry(None, 400, attempt=0, max_retries=3) is False         # 4xx 不可重试
assert should_retry('timeout', None, attempt=3, max_retries=3) is False   # 用尽
assert should_retry('bad_param', None, attempt=0, max_retries=3) is False # 其它不可重试
print('✅ 练习 1 通过：超时/5xx 重试、4xx/用尽/其它不重试')

## ✏️ 练习 2：带抖动的指数退避序列

退避要 `base*2**k` 指数增长 + **随机抖动**打散重试时刻(防惊群)。

实现 `backoff_sequence(base, n, cap, jitter_fn)`：返回前 `n` 次重试的等待时间列表，第 k 次(从 0 起) = `min(base * 2**k, cap)` 再 `* (1 + jitter_fn(k))`；`cap` 是封顶。`jitter_fn(k)` 返回 [0,1) 的抖动(测试传确定性函数)。

In [ ]:
def backoff_sequence(base, n, cap, jitter_fn):
    # TODO: 返回 [min(base*2**k, cap) * (1 + jitter_fn(k)) for k in range(n)]
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
no_jitter = lambda k: 0.0
seq = backoff_sequence(1.0, 5, cap=8.0, jitter_fn=no_jitter)
assert seq == [1.0, 2.0, 4.0, 8.0, 8.0]      # 指数增长, 第 4 次被 cap 封顶
# 带固定抖动 0.5 -> 每个 *1.5
seq2 = backoff_sequence(1.0, 3, cap=100.0, jitter_fn=lambda k: 0.5)
assert seq2 == [1.5, 3.0, 6.0]
print('无抖动退避:', seq, '\n带抖动退避:', seq2)
print('✅ 练习 2 通过：指数退避 + 封顶 + 抖动(打散重试防惊群)')

## ✏️ 练习 3：优雅停机——把 running 任务安全收尾

停机时不能粗暴 kill(会丢 running 任务)。**优雅停机**：停止取新任务、把 running 的回退到 queued、再返回剩余工作量。

实现 `graceful_shutdown(tasks)`：把所有 `state=='running'` 的任务转回 `'queued'`(回退重跑)，返回 `{'requeued': 被回退的 id 列表, 'pending': 停机后仍待处理(queued)的总数}`。

In [ ]:
def graceful_shutdown(tasks):
    # TODO: 遍历 tasks:
    #   state=='running' -> 改回 'queued', 记入 requeued
    #   返回 {'requeued': [...], 'pending': state=='queued' 的总数}
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
class T:
    def __init__(self, tid, state): self.id, self.state = tid, state
tasks = {'a': T('a', 'running'), 'b': T('b', 'queued'),
         'c': T('c', 'succeeded'), 'd': T('d', 'running')}
out = graceful_shutdown(tasks)
assert set(out['requeued']) == {'a', 'd'}      # 两个 running 被回退
assert tasks['a'].state == 'queued' and tasks['d'].state == 'queued'
assert tasks['c'].state == 'succeeded'         # 已完成的不动
assert out['pending'] == 3                      # 原 b + 回退的 a,d
print('优雅停机:', out)
print('✅ 练习 3 通过：running 任务安全回退到 queued, 重启后可继续(不丢任务)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def should_retry(error_type, status_code, attempt, max_retries):
    if attempt >= max_retries:
        return False
    if error_type == 'timeout':
        return True
    if status_code is not None and 500 <= status_code <= 599:
        return True
    if status_code is not None and 400 <= status_code <= 499:
        return False
    return False

In [ ]:
# 练习 2 参考答案
def backoff_sequence(base, n, cap, jitter_fn):
    return [min(base * 2 ** k, cap) * (1 + jitter_fn(k)) for k in range(n)]

In [ ]:
# 练习 3 参考答案
def graceful_shutdown(tasks):
    requeued = []
    for tid, t in tasks.items():
        if t.state == 'running':
            t.state = 'queued'
            requeued.append(tid)
    pending = sum(1 for t in tasks.values() if t.state == 'queued')
    return {'requeued': requeued, 'pending': pending}

---
## 🧪 真实数据胶囊：把全栈(01-05)拧成一个 agent 服务

全课合流：一个『研究 agent 服务』收到任务，全栈流转——**入队(05)→worker(05)→编排(02)→subagent(01)→权限(03)→可观测(04)→收尾(05)**。

下面用前面各模块的零件(简化版)端到端跑一个任务，看六个模块如何拼成一个有机系统。

> 形状对照：这正是 `05_deploy/agent_cli.py` 想升级成的服务形态——把单 agent 的 run_agent 套进队列+状态机+重试+持久的运行时。

In [ ]:
# 六模块零件(极简版)拼一个 agent 服务
def m02_decompose(task):              # 02 编排: 分解
    return [f'调研 {co}公司' for co in ['A', 'B', 'C']]
def m01_subagent(subtask):            # 01 subagent: 隔离上下文跑子任务
    data = {'调研 A公司': 'A +10%', '调研 B公司': 'B +12%', '调研 C公司': 'C -3%'}
    return {'task': subtask, 'status': 'ok', 'result': data[subtask], 'tokens': 100}
def m03_permit(call):                 # 03 权限: 只读放行(检索 agent 无危险权限)
    return call['name'] in {'web_search', 'read_file'}
def m02_synthesize(results):          # 02 合成
    return '对比: ' + '；'.join(r['result'] for r in results if r['status'] == 'ok')

def research_agent(job):
    '''一个完整的研究 agent: 编排+subagent+权限+可观测, 作为 worker 的 handler。'''
    trace = []                                   # 04 可观测: 简化 span 记录
    assert m03_permit({'name': 'web_search'})     # 03 权限把关
    subtasks = m02_decompose(job)                 # 02 分解
    results = []
    for st in subtasks:
        r = m01_subagent(st)                      # 01 派 subagent
        trace.append({'span': st, 'tokens': r['tokens']})
        results.append(r)
    answer = m02_synthesize(results)              # 02 合成
    total_tokens = sum(s['tokens'] for s in trace)  # 04 成本聚合
    return {'answer': answer, 'tokens': total_tokens, 'n_subagents': len(results)}

# 05 部署: 把这个 agent 放进服务的 worker 跑
service = AgentService()
service.submit('research-1', '对比 A、B、C 三家公司', priority=1)
service.run_until_empty(research_agent)
task = service.tasks['research-1']
print('任务状态:', task.state)
print('结果:', task.result['answer'])
print('派了', task.result['n_subagents'], '个 subagent, 共', task.result['tokens'], 'token')
# 全栈验证: 入队->跑->成功; 编排派了 3 个 subagent; 成本被聚合
assert task.state == 'succeeded'
assert task.result['n_subagents'] == 3
assert 'A +10%' in task.result['answer'] and 'C -3%' in task.result['answer']
assert task.result['tokens'] == 300
# 健康检查通过
assert health_check(service.tasks, service.queue)['ready'] is True
print('✅ 全栈拧成一个 agent 服务：队列+worker+编排+subagent+权限+可观测 端到端跑通')

**🧪 胶囊练习**：实现 `service_summary(tasks)`：给定服务的任务表，返回 `{'total':总数, 'succeeded':成功数, 'dead':死信数, 'success_rate':成功率(成功/总数, 保留2位)}`。（运维 dashboard 就这样汇总一个 agent 服务的整体健康。）

In [ ]:
def service_summary(tasks):
    # TODO: 统计 total/succeeded/dead, success_rate=round(succeeded/total,2)(total=0 时为 0.0)
    raise NotImplementedError

In [ ]:
# 自测
class TT:
    def __init__(self, s): self.state = s
tasks = {'a': TT('succeeded'), 'b': TT('succeeded'), 'c': TT('dead'), 'd': TT('running')}
s = service_summary(tasks)
assert s['total'] == 4 and s['succeeded'] == 2 and s['dead'] == 1
assert s['success_rate'] == 0.5
print('服务汇总:', s)
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def service_summary(tasks):
    total = len(tasks)
    succ = sum(1 for t in tasks.values() if t.state == 'succeeded')
    dead = sum(1 for t in tasks.values() if t.state == 'dead')
    return {'total': total, 'succeeded': succ, 'dead': dead,
            'success_rate': round(succ / total, 2) if total else 0.0}

---
## 🔧 旁注：真实部署 + 真实 Claude（无 key 回退）

本课的内存队列 / worker / 重试 / 持久，换成真实生产只是替换后端；agent 本身用真实 Claude（伪代码，**本环境用 MockLLM；有 key 走真实 claude-opus-4-8**）：

```python
# 真实部署: 队列换 Redis/Celery、持久换 Postgres、worker 多进程; 控制流不变
import os
def make_agent_handler():
    if os.environ.get('ANTHROPIC_API_KEY'):
        import anthropic
        client = anthropic.Anthropic()
        def handler(job):                      # 一个任务 = 一次(可多步)真实 agent 运行
            resp = client.messages.create(model='claude-opus-4-8', max_tokens=2048,
                                          messages=[{'role':'user','content':job}],
                                          tools=SCHEMAS)               # 03 权限裹在工具执行外
            return {'answer': ''.join(b.text for b in resp.content if b.type=='text'),
                    'tokens': resp.usage.input_tokens + resp.usage.output_tokens}  # 04 真实成本
        return handler
    return research_agent                       # 无 key -> 回退到本课的离线 agent, 服务照常

# worker: while True: task = queue.get(); run_with_retry(task, make_agent_handler())
# 这正是 05_deploy/agent_cli.py 的 run_agent 套进部署运行时的形态
```

对应关系：`AgentService` ↔ 真实服务、`TaskQueue` ↔ Redis/SQS、`snapshot/restore` ↔ DB、`run_with_retry` ↔ Celery 重试。你练的队列/状态机/重试/持久/健康**控制流原样适用**，换成真实后端 + 真实 claude-opus-4-8（无 key 自动回退）即可上生产。这就是「scaffold 可迁移」。

### 小结
- 部署 = 给 agent 装上能 **7×24 自动运转**的运行时；假设最坏情况一定发生、为每种坏情况备好应对。
- **任务队列**：解耦接收与执行(削峰/扩容/重试)；『至少一次』投递逼出**幂等**。
- **状态机**：合法转移管理任务一生、拒绝非法转移；状态变迁可观测。
- **重试退避**：先区分可重试(瞬时)/不可重试(确定性)；可重试用指数退避+抖动+上限，用尽进死信。
- **持久化**：崩溃不丢任务；恢复时 running 孤儿任务重新入队等**幂等**重跑。
- **幂等/健康检查/优雅停机**：重复执行安全、外部探死活、停机不丢任务。
- **全栈合流**：队列+worker+编排(02)+subagent(01)+权限(03)+可观测(04) = 一个能上生产的 agent 服务。

🎓 **全课完结**：你已从零把多 agent 系统搭起(01-02)、控住(03)、看清(04)、部署进生产(05)。把 `MockLLM` 换成真实 `claude-opus-4-8`(无 key 自动回退)，你写的整套系统就能上真实世界——你练的是 **agent 系统工程**，模型只是一个可替换的零件。